<a href="https://colab.research.google.com/github/tadapin/tidb-colab-tutorials/blob/main/tutorials/10_image_search.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# 画像検索 / CLIP + pytidb

このノートブックは **pytidb シリーズの第 10 回** です。OpenAI の CLIP (ViT-B/32) を使って画像を 512 次元のベクトルに変換し、TiDB で類似画像を検索します。

参考: [tidb-vector-python `image_search/example.ipynb`](https://github.com/pingcap/tidb-vector-python/blob/main/examples/image_search/example.ipynb) を pytidb ベースに書き直したものです。

## 学習目標
- CLIP で **画像とテキストを同じ 512 次元ベクトル空間**に埋め込む
- 独自 `BaseEmbeddingFunction` をマルチモーダル (`source_type="image"` / `"text"`) に対応させる
- 画像ベクトルを TiDB の `VECTOR(512)` に保存する
- **テキスト→画像** 検索 (例: "dog" で犬の写真を引く)
- **画像→画像** 検索 (クエリ画像から類似画像を引く)

## 注意
- CLIP ViT-B/32 は **英語**のテキストに最適化されたモデルです。クエリ文は英語で書いてください (日本語は精度が大きく落ちます)
- 初回のみモデル DL が走ります (約 150 MB)、Colab の無料枠 (CPU ランタイム) で数十秒〜1 分程度
- API キーは不要

前提: `09_custom_embedding.ipynb` を実行済みだと構造が分かりやすいです。


## 1. 依存パッケージのインストール

追加で `transformers`・`datasets`・`pillow` を入れます (`torch` は Colab に同梱)。


In [ ]:
!pip install -q pytidb transformers datasets pillow


## 2. TiDB Cloud Zero の払い出し


In [ ]:
from pathlib import Path
import requests

# TiDB Cloud Zero のAPIエンドポイント (サインアップ不要、30日有効)
ZERO_API = "https://zero.tidbapi.com/v1beta1/instances"


def request_zero_instance(tag: str = "pytidb-tutorial") -> dict:
    """TiDB Cloud Zero のインスタンスを作成して接続情報を返す"""
    resp = requests.post(ZERO_API, json={"tag": tag}, timeout=30)
    resp.raise_for_status()
    return resp.json()["instance"]


instance = request_zero_instance(tag="pytidb-image-search")
conn = instance["connection"]
claim_url = instance.get("claimInfo", {}).get("claimUrl", "(取得失敗)")
expires_at = instance.get("expiresAt", "(取得失敗)")

print("=== プロビジョニング完了 ===")
print(f"Host     : {conn['host']}")
print(f"User     : {conn['username']}")
print(f"Expires  : {expires_at}")
print()
print("インスタンスは 30 日後に自動削除されます。")
print("保持したい場合は claim URL を開いてください:")
print(claim_url)


## 3. CLIP モデルを読み込む + pytidb 用のマルチモーダル埋め込みクラス

`BaseEmbeddingFunction` を継承し、`source_type` に応じて画像・テキストの両方に対応します。
内部で `transformers` の `CLIPModel.get_image_features` / `get_text_features` を呼ぶだけ。


In [ ]:
from typing import Any, List, Optional
import torch
from PIL import Image
from transformers import CLIPModel, CLIPProcessor
from pytidb.embeddings.base import BaseEmbeddingFunction

CLIP_MODEL = "openai/clip-vit-base-patch32"
CLIP_DIM = 512


class CLIPEmbedding(BaseEmbeddingFunction):
    model_config = {"arbitrary_types_allowed": True}

    def __init__(self, model_name: str = CLIP_MODEL, **kwargs):
        super().__init__(
            provider="clip",
            model_name=model_name,
            dimensions=CLIP_DIM,
            use_server=False,
            **kwargs,
        )
        # HF からモデルと processor をロード (初回のみ DL)
        self.__dict__["_model"] = CLIPModel.from_pretrained(model_name)
        self.__dict__["_processor"] = CLIPProcessor.from_pretrained(model_name)

    @staticmethod
    def _to_tensor(out):
        # transformers のバージョンによっては tensor ではなく出力オブジェクトが返る
        if hasattr(out, "cpu"):
            return out
        for attr in ("text_embeds", "image_embeds", "pooler_output", "last_hidden_state"):
            v = getattr(out, attr, None)
            if v is not None and hasattr(v, "cpu"):
                return v
        raise TypeError(f"Cannot extract tensor from CLIP output: {type(out).__name__}")

    def _encode_text(self, texts: List[str]) -> List[List[float]]:
        proc = self.__dict__["_processor"]
        model = self.__dict__["_model"]
        with torch.no_grad():
            inputs = proc(text=texts, return_tensors="pt", padding=True, truncation=True)
            features = self._to_tensor(model.get_text_features(**inputs))
        return [row.tolist() for row in features.cpu().numpy()]

    def _encode_images(self, images) -> List[List[float]]:
        proc = self.__dict__["_processor"]
        model = self.__dict__["_model"]
        with torch.no_grad():
            inputs = proc(images=images, return_tensors="pt")
            features = self._to_tensor(model.get_image_features(**inputs))
        return [row.tolist() for row in features.cpu().numpy()]

    def get_query_embedding(self, query: Any, source_type: Optional[str] = "text", **kwargs) -> List[float]:
        if source_type == "image":
            return self._encode_images([query])[0]
        return self._encode_text([str(query)])[0]

    def get_source_embedding(self, source: Any, source_type: Optional[str] = "text", **kwargs) -> List[float]:
        if source_type == "image":
            return self._encode_images([source])[0]
        return self._encode_text([str(source)])[0]

    def get_source_embeddings(self, sources: List[Any], source_type: Optional[str] = "text", **kwargs) -> List[List[float]]:
        if source_type == "image":
            return self._encode_images(sources)
        return self._encode_text([str(s) for s in sources])


embed_fn = CLIPEmbedding()
print("CLIP ready  dim =", embed_fn.dimensions)


## 4. テーブルを作成する

画像を直接 TiDB に格納する代わりに、**画像 ID とベクトル**だけ保存して、画像本体は Python 側のリストから参照する構成にします (参考ノートブックと同じ)。


In [ ]:
from pytidb import TiDBClient
from pytidb.schema import Field, TableModel, VectorField

db = TiDBClient.connect(
    host=conn["host"],
    port=4000,
    username=conn["username"],
    password=conn["password"],
    database="image_search_demo",
    ensure_db=True,
)


class ImageVec(TableModel):
    __tablename__ = "image_vectors"
    __table_args__ = {"extend_existing": True}

    id: int = Field(primary_key=True)
    image_id: int = Field()         # 画像本体はメモリ中のリスト image_pool[image_id]
    label: str = Field()            # ImageNet ラベル (可読用)
    embedding: list[float] = VectorField(dimensions=CLIP_DIM)


table = db.create_table(schema=ImageVec, if_exists="overwrite")
print("テーブル準備OK:", table.table_name)


## 5. サンプル画像をロードする

`datasets` から ImageNet tiny を取得します (参考ノートブックと同じ `theodor1289/imagenet-1k_tiny`)。
先頭 20 枚だけ使います。


In [ ]:
import datasets

ds = datasets.load_dataset("theodor1289/imagenet-1k_tiny", split="train")
image_pool = []        # [PIL.Image]
labels_pool = []       # [str]
for i, row in enumerate(ds):
    if i >= 20:
        break
    image_pool.append(row["image"].convert("RGB"))
    # 列名はデータセットによって違う可能性あり
    label = row.get("label", row.get("fine_label", "unknown"))
    labels_pool.append(str(label))

print(f"loaded {len(image_pool)} images")


### 画像をインラインで確認する


In [ ]:
import matplotlib.pyplot as plt

def show_grid(images, titles=None, cols=5, width=2):
    n = len(images)
    rows = (n + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(cols * width, rows * width))
    axes = axes.flatten() if n > 1 else [axes]
    for ax in axes:
        ax.axis("off")
    for i, img in enumerate(images):
        axes[i].imshow(img)
        if titles:
            axes[i].set_title(titles[i], fontsize=9)
    plt.tight_layout()
    plt.show()


show_grid(image_pool[:20], titles=[f"#{i}" for i in range(len(image_pool[:20]))])


## 6. 画像をベクトル化して TiDB に保存する

`embed_fn.get_source_embeddings(images, source_type="image")` で一括エンコードします。


In [ ]:
image_vectors = embed_fn.get_source_embeddings(image_pool, source_type="image")

rows = [
    ImageVec(id=i + 1, image_id=i, label=labels_pool[i], embedding=image_vectors[i])
    for i in range(len(image_pool))
]
table.bulk_insert(rows)
print(f"投入完了: {table.rows()} 件")


## 7. テキスト → 画像 検索

検索クエリ文字列を CLIP の **テキストエンコーダ**に通すと、画像ベクトルと同じ空間の 512 次元ベクトルが得られます。
それを `table.search(vec)` に直接渡すだけ。


In [ ]:
QUERIES = ["a photo of a dog", "sushi on a plate", "a classic sports car"]

for q in QUERIES:
    qvec = embed_fn.get_query_embedding(q, source_type="text")
    hits = table.search(qvec).limit(5).to_pydantic()
    print(f"\n=== query: {q!r} ===")
    imgs = [image_pool[h.hit.image_id] for h in hits]
    titles = [f"sim={h.similarity_score:.3f}\n[{h.hit.label}]" for h in hits]
    show_grid(imgs, titles=titles, cols=5)


## 8. 画像 → 画像 検索

CLIP の画像エンコーダで得たベクトルを検索側に渡せば、**類似画像検索**になります。
クエリ画像はデータセット内の 1 枚を流用しても、外部 URL からロードしても OK。


In [ ]:
# データセット先頭の 1 枚をクエリにする
query_image = image_pool[0]
print("Query image:")
show_grid([query_image], titles=["query"], cols=1)

qvec = embed_fn.get_query_embedding(query_image, source_type="image")
hits = table.search(qvec).limit(5).to_pydantic()

print("\n=== top-5 similar images ===")
imgs = [image_pool[h.hit.image_id] for h in hits]
titles = [f"sim={h.similarity_score:.3f}\n[{h.hit.label}]" for h in hits]
show_grid(imgs, titles=titles, cols=5)


## 追加実験

- `QUERIES` に自分で考えた英語フレーズを追加して上位 5 枚がどう動くか確認
- 外部画像 URL を `requests.get` + `PIL.Image.open(io.BytesIO(...))` で読み込み、画像→画像検索のクエリにする
- `limit(10)` にしたり `distance_threshold(0.5)` を挟んで、閾値で絞ってみる
- CLIP を **`openai/clip-vit-large-patch14`** (約 900 MB、768 次元) に差し替えて精度の違いを比較 (ただし Colab 無料枠だと重め)
- 多言語で検索したい場合は `sentence-transformers/clip-ViT-B-32-multilingual-v1` に置き換えて再構築
